In [4]:
import joblib
from dataclasses import dataclass, field
from typing import Tuple, Dict, Any

In [5]:
@dataclass(frozen=True)
class FinalCfg:
    test_size: float = 0.2
    random_state: int = 42
    oof_splits: int = 5
    weight_win: float = 2.0
    weight_loss: float = 1.0
    drop_non_features: Tuple[str, ...] = ('game_id','home_team','away_team','season','week')
    categorical_cols: Tuple[str, ...] = ('roof','surface')
    boolean_cols: Tuple[str, ...] = (
        'is_playoff','is_final_week','home_qb_switch','away_qb_switch','is_home_qb_new','is_away_qb_new'
    )
    # For OOF weighting model
    base_xgb_params: Dict[str, Any] = field(default_factory=lambda: dict(
        n_estimators=500, max_depth=3, learning_rate=0.01, min_child_weight=3,
        subsample=0.6, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=3.0,
        objective='reg:squarederror', random_state=42, tree_method='hist', n_jobs=1
    ))

In [6]:
res = joblib.load('/content/drive/MyDrive/BettingEdgeContinued/fantasy_model.pkl')

In [7]:
print(res.keys())
# → dict_keys(['mae', 'r2', 'y_test', 'y_pred', 'pipeline', 'used_columns', 'config'])

dict_keys(['mae', 'r2', 'y_test', 'y_pred', 'pipeline', 'used_columns', 'config'])


In [10]:
# The pipeline is what actually makes predictions
pipeline = res["pipeline"]
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['roof', 'surface']),
                                                 ('num', 'passthrough',
                                                  ['spread_line', 'away_rest',
                                                   'home_rest', 'total_line',
                                                   'div_game', 'temp', 'wind',
                                                   'home_rolling_avg_epa',
                                                   'home_rolling_avg_yards',
                                                   'home_rolling_play_count',
                                                   'away_rolling_avg_e...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.01,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=2, max_leaves=None, min_child_weight=3,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=384, n_jobs=1,
                              num_parallel_tree=None, ...))])

In [11]:
# This tells you EXACTLY what columns the model was trained on
pre = pipeline.named_steps["preprocessor"]
pre

ColumnTransformer(transformers=[('cat',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['roof', 'surface']),
                                ('num', 'passthrough',
                                 ['spread_line', 'away_rest', 'home_rest',
                                  'total_line', 'div_game', 'temp', 'wind',
                                  'home_rolling_avg_epa',
                                  'home_rolling_avg_yards',
                                  'home_rolling_play_count',
                                  'away_rolling_avg_epa',
                                  'away_rolling_avg_yards',
                                  'aw...
                                  'epa_home_def_away_off_rolling_diff',
                                  'avg_yards_home_off_away_def_rolling_diff',
                                  'avg_yards_home_def_away_off_rolling_diff',
                                  'play_count_home_off_away_def_rolling_diff',
                                  'play_count_home_def_away_off_rolling_diff',
                                  'home_recent_sos_opponent_avg',
                                  'home_season_sos_opponent_avg',
                                  'away_recent_sos_opponent_avg',
                                  'away_season_sos_opponent_avg', 'sos_diff', ...])],
                  verbose_feature_names_out=False)

In [12]:
cat_cols  = pre.transformers_[0][2]   # OrdinalEncoder columns (roof, surface)
num_cols  = pre.transformers_[1][2]   # passthrough columns (everything else)

In [13]:
print("Categorical features:", cat_cols)
print("Numeric/Boolean features:", num_cols)
print("\nTotal features expected:", len(cat_cols) + len(num_cols))

Categorical features: ['roof', 'surface']
Numeric/Boolean features: ['spread_line', 'away_rest', 'home_rest', 'total_line', 'div_game', 'temp', 'wind', 'home_rolling_avg_epa', 'home_rolling_avg_yards', 'home_rolling_play_count', 'away_rolling_avg_epa', 'away_rolling_avg_yards', 'away_rolling_play_count', 'home_rolling_allowed_avg_epa', 'home_rolling_allowed_avg_yards', 'home_rolling_allowed_play_count', 'away_rolling_allowed_avg_epa', 'away_rolling_allowed_avg_yards', 'away_rolling_allowed_play_count', 'epa_home_off_away_def_rolling_diff', 'epa_home_def_away_off_rolling_diff', 'avg_yards_home_off_away_def_rolling_diff', 'avg_yards_home_def_away_off_rolling_diff', 'play_count_home_off_away_def_rolling_diff', 'play_count_home_def_away_off_rolling_diff', 'home_recent_sos_opponent_avg', 'home_season_sos_opponent_avg', 'away_recent_sos_opponent_avg', 'away_season_sos_opponent_avg', 'sos_diff', 'season_sos_diff', 'home_allpro_last_3_years_weighted', 'away_allpro_last_3_years_weighted', 'diff

In [15]:
import nflreadpy as nfl
import pandas as pd

raw = nfl.load_schedules([2025])

print(type(raw))   # confirms it's a polars DataFrame

# Convert Polars → Pandas correctly
schedule = raw.to_pandas()

print(schedule.shape)
print(schedule.columns.tolist())

<class 'polars.dataframe.frame.DataFrame'>
(285, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [16]:
key_cols = [
    'game_id', 'home_team', 'away_team', 'week', 'season',
    'spread_line', 'total_line', 'away_rest', 'home_rest',
    'div_game', 'roof', 'surface', 'temp', 'wind', 'result'
]

for col in key_cols:
    status = "✅" if col in schedule.columns else "❌ MISSING"
    print(f"{status}  {col}")

print("\n\nAll available columns:")
print(schedule.columns.tolist())

✅  game_id
✅  home_team
✅  away_team
✅  week
✅  season
✅  spread_line
✅  total_line
✅  away_rest
✅  home_rest
✅  div_game
✅  roof
✅  surface
✅  temp
✅  wind
✅  result


All available columns:
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [18]:
TARGET_WEEK = 10  # change this each week you want to predict

week_games = schedule[schedule['week'] == TARGET_WEEK]

# Show just the columns relevant to us
cols = ['game_id', 'home_team', 'away_team', 'gameday', 'gametime',
        'spread_line', 'total_line', 'away_rest', 'home_rest',
        'div_game', 'roof', 'surface', 'temp', 'wind']

print(week_games[cols].to_string())

             game_id home_team away_team     gameday gametime  spread_line  total_line  away_rest  home_rest  div_game      roof     surface  temp  wind
135   2025_10_LV_DEN       DEN        LV  2025-11-06    20:15          9.5        42.5          4          4         1  outdoors       grass  60.0  10.0
136  2025_10_ATL_IND       IND       ATL  2025-11-09    09:30          6.5        48.5          7          7         0    closed       grass  46.0   2.0
137   2025_10_NO_CAR       CAR        NO  2025-11-09    13:00          5.5        38.5          7          7         1  outdoors       grass  73.0  15.0
138  2025_10_NYG_CHI       CHI       NYG  2025-11-09    13:00          4.5        45.5          7          7         0  outdoors       grass  33.0  10.0
139  2025_10_JAX_HOU       HOU       JAX  2025-11-09    13:00         -1.5        37.5          7          7         1    closed   astroturf   NaN   NaN
140  2025_10_BUF_MIA       MIA       BUF  2025-11-09    13:00         -8.5        

In [19]:
# All completed games BEFORE the target week — used to build rolling features
history = schedule[
    (schedule['week'] < TARGET_WEEK) &
    (schedule['result'].notna())   # result is NaN for games not yet played
].copy()

# The games you actually want to predict
upcoming = schedule[schedule['week'] == TARGET_WEEK].copy()

print(f"History: {len(history)} completed games (weeks 1 through {TARGET_WEEK - 1})")
print(f"Upcoming: {len(upcoming)} games to predict in week {TARGET_WEEK}")

History: 135 completed games (weeks 1 through 9)
Upcoming: 14 games to predict in week 10


In [21]:
# These are the Group 1 features the model expects
group1_features = [
    'spread_line', 'away_rest', 'home_rest', 'total_line',
    'div_game', 'temp', 'wind', 'roof', 'surface'
]

print("Group 1 feature check for upcoming games:\n")
print(upcoming[group1_features].to_string())
print("\nNull counts:")
print(upcoming[group1_features].isnull().sum())

Group 1 feature check for upcoming games:

     spread_line  away_rest  home_rest  total_line  div_game  temp  wind      roof     surface
135          9.5          4          4        42.5         1  60.0  10.0  outdoors       grass
136          6.5          7          7        48.5         0  46.0   2.0    closed       grass
137          5.5          7          7        38.5         1  73.0  15.0  outdoors       grass
138          4.5          7          7        45.5         0  33.0  10.0  outdoors       grass
139         -1.5          7          7        37.5         1   NaN   NaN    closed   astroturf
140         -8.5          7         10        50.5         1  84.0   6.0  outdoors       grass
141         -4.5         10          7        48.5         0   NaN   NaN      dome   sportturf
142         -1.5         14         14        37.5         0  62.0  12.0  outdoors   fieldturf
143          2.5          7         14        48.5         0  82.0  11.0  outdoors       grass
144    